# NYC Yellow Taxi Trip Preprocessing (January-March 2025)

This notebook prepares valid Yellow Taxi trips for hourly pickup-demand modelling. The three monthly Parquet files are compared before they are concatenated. The combined data is inspected once, after which each cleaning decision follows the findings from that inspection.

The notebook keeps the original trip variables long enough to check trip validity. Only the pickup time and pickup zone are used to create the final hourly demand target.


## 1. Import Libraries and Define Paths

The study covers pickups from 1 January through 31 March 2025. The ending boundary of 1 April is exclusive.


In [12]:
from pathlib import Path
from IPython.display import display
import gc
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(
    r"C:\Asia Pacific University\All Module Notes\Semister-5\Investigations"
    r"\FYP Semester 1\Progress\fyp"
)
RAW_DIR = PROJECT_ROOT / "all_raw_data" / "tlc_trips"
ZONE_PATH = PROJECT_ROOT / "processed_outputs" / "taxi_zone" / "nyc_taxi_zones_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "processed_outputs" / "yellow_taxi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_START = pd.Timestamp("2025-01-01 00:00:00", tz="America/New_York")
STUDY_END = pd.Timestamp("2025-04-01 00:00:00", tz="America/New_York")
LOCAL_TIMEZONE = "America/New_York"

MONTH_FILES = {
    "2025-01": RAW_DIR / "yellow_tripdata_2025-01.parquet",
    "2025-02": RAW_DIR / "yellow_tripdata_2025-02.parquet",
    "2025-03": RAW_DIR / "yellow_tripdata_2025-03.parquet",
}


## 2. Load and Check the Cleaned Taxi-Zone Data


In [13]:
required_paths = [*MONTH_FILES.values(), ZONE_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(map(str, missing_paths)))

zone_reference = pd.read_csv(ZONE_PATH)
display(zone_reference.head())
print(f"Taxi-zone rows: {len(zone_reference):,}")
print(f"Taxi-zone columns: {zone_reference.shape[1]}")
print(f"Unique LocationID values: {zone_reference['LocationID'].nunique():,}")
print("Boroughs:", sorted(zone_reference["borough"].dropna().astype(str).unique()))

valid_zone_ids = pd.Index(
    sorted(pd.to_numeric(zone_reference["LocationID"], errors="raise").astype(int).unique()),
    name="LocationID",
)


,LocationID,borough,zone,service_zone
0,2,Queens,Jamaica Bay,Boro Zone
1,3,Bronx,Allerton/Pelham Gardens,Boro Zone
2,4,Manhattan,Alphabet City,Yellow Zone
3,5,Staten Island,Arden Heights,Boro Zone
4,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone


Taxi-zone rows: 262
Taxi-zone columns: 4
Unique LocationID values: 262
Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


**Interpretation.** The cleaned reference should contain 262 unique taxi zones across the five NYC boroughs. These IDs will be used only to check pickup and drop-off locations.


## 3. Load, Compare and Combine the Monthly Trip Files

### 3.1 Load the Files Separately and Compare Their Structures

Each file is loaded separately first. Its row count, column count, column names and data types are compared with the other months before concatenation.


In [14]:
monthly_frames = {}

for source_month, path in MONTH_FILES.items():
    monthly_frames[source_month] = pd.read_parquet(path)

print("The three monthly Parquet files have been loaded separately.")


The three monthly Parquet files have been loaded separately.


#### Check: compare the monthly structures

Before concatenation, compare the record count, number of variables, column names, column order and data types in all three files.


In [15]:
structure_rows = []
reference_columns = monthly_frames["2025-01"].columns.tolist()
reference_dtypes = monthly_frames["2025-01"].dtypes.astype(str).to_dict()

for source_month, monthly in monthly_frames.items():
    columns = monthly.columns.tolist()
    dtypes = monthly.dtypes.astype(str).to_dict()
    structure_rows.append({
        "source_month": source_month,
        "records": len(monthly),
        "variables": monthly.shape[1],
        "same_column_names_and_order": columns == reference_columns,
        "same_data_types": dtypes == reference_dtypes,
    })

structure_summary = pd.DataFrame(structure_rows)
display(structure_summary)

schema_summary = pd.DataFrame({
    "variable": reference_columns,
    "data_type": [reference_dtypes[column] for column in reference_columns],
})
display(schema_summary)


,source_month,records,variables,same_column_names_and_order,same_data_types
0,2025-01,3475226,20,True,True
1,2025-02,3577543,20,True,True
2,2025-03,4145257,20,True,True


,variable,data_type
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[us]
2,tpep_dropoff_datetime,datetime64[us]
3,passenger_count,float64
4,trip_distance,float64
5,RatecodeID,float64
6,store_and_fwd_flag,object
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


**Interpretation.** Concatenation is appropriate only when the monthly files have matching structures. The schema table shows exactly which variables and data types are present in every file.


### 3.2 Add the Source Month and Concatenate the Files

`source_month` records which Parquet file supplied each row. It is an audit field and is different from the modelling `month` feature created later from the actual pickup timestamp.


In [16]:
for source_month, monthly in monthly_frames.items():
    monthly["source_month"] = source_month

trips_all = pd.concat(monthly_frames.values(), ignore_index=True)
del monthly_frames
gc.collect()

print(f"Combined records: {len(trips_all):,}")
print(f"Combined variables: {trips_all.shape[1]}")
display(trips_all.head())


Combined records: 11,198,026
Combined variables: 21


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,source_month
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,2025-01
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2025-01
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,2025-01
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,2025-01
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,2025-01


**Interpretation.** The files are stacked vertically, so the original variables remain as columns while January, February and March trips appear as additional rows. The new `source_month` field preserves the file of origin.


### 3.3 Inspect the Combined Dataset Once


#### Check: combined dimensions, columns and data types


In [17]:
print(f"Records: {len(trips_all):,}")
print(f"Columns: {trips_all.shape[1]}")
print("Column names:")
print(trips_all.columns.tolist())
trips_all.info(show_counts=True)


Records: 11,198,026
Columns: 21
Column names:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'source_month']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11198026 entries, 0 to 11198025
Data columns (total 21 columns):
 #   Column                 Non-Null Count     Dtype         
---  ------                 --------------     -----         
 0   VendorID               11198026 non-null  int32         
 1   tpep_pickup_datetime   11198026 non-null  datetime64[us]
 2   tpep_dropoff_datetime  11198026 non-null  datetime64[us]
 3   passenger_count        8934277 non-null   float64       
 4   trip_distance          11198026 non-null  float64       
 5   RatecodeID             8934277 non-

#### Check: missing values


In [18]:
missing_summary = (
    trips_all.isna().sum()
    .rename("missing_records")
    .to_frame()
    .assign(missing_percentage=lambda x: x["missing_records"] / len(trips_all) * 100)
    .sort_values("missing_records", ascending=False)
)
display(missing_summary)


,missing_records,missing_percentage
passenger_count,2263749,20.215608
Airport_fee,2263749,20.215608
congestion_surcharge,2263749,20.215608
store_and_fwd_flag,2263749,20.215608
RatecodeID,2263749,20.215608
trip_distance,0,0.000000
tpep_dropoff_datetime,0,0.000000
tpep_pickup_datetime,0,0.000000
VendorID,0,0.000000
payment_type,0,0.000000


#### Check: numerical ranges


In [19]:
numeric_summary = trips_all.select_dtypes(include="number").describe().T
display(numeric_summary)


,count,mean,std,min,25%,50%,75%,max
VendorID,11198026.0,1.803277,0.482393,1.00,2.00,2.00,2.00,7.00
passenger_count,8934277.0,1.288654,0.735222,0.00,1.00,1.00,1.00,9.00
trip_distance,11198026.0,6.179361,581.823197,0.00,1.00,1.72,3.24,320136.29
RatecodeID,8934277.0,2.453865,11.513329,1.00,1.00,1.00,1.00,99.00
PULocationID,11198026.0,163.314009,65.414264,1.00,125.00,161.00,233.00,265.00
DOLocationID,11198026.0,162.527806,69.830883,1.00,113.00,162.00,234.00,265.00
payment_type,11198026.0,0.976855,0.723082,0.00,1.00,1.00,1.00,5.00
fare_amount,11198026.0,17.241468,261.991719,-1807.60,8.60,12.80,20.50,863372.12
extra,11198026.0,1.256459,1.851653,-9.25,0.00,0.00,2.50,22.55
mta_tax,11198026.0,0.479046,0.134748,-0.50,0.50,0.50,0.50,10.50


#### Check: exact duplicate trip rows


In [20]:
original_columns = [column for column in trips_all.columns if column != "source_month"]
duplicate_mask = trips_all.duplicated(subset=original_columns, keep="first")
duplicate_count = int(duplicate_mask.sum())
print(f"Exact duplicate trip rows: {duplicate_count:,}")


Exact duplicate trip rows: 0


**Interpretation.** The combined inspection provides the overall missing-value, range and duplicate findings.


## 4. Select the Trip-Validity and Optional Diagnostic Columns

Eight TLC variables define a valid trip. `VendorID`, `RatecodeID` and `total_amount` are retained temporarily to investigate unusual records without automatically excluding them.


In [21]:
working_columns = [
    "source_month",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "VendorID",
    "RatecodeID",
    "total_amount",
]

trips = trips_all[working_columns].copy()
del trips_all
gc.collect()

print("Working columns:")
print(working_columns)
print(f"Working records: {len(trips):,}")


Working columns:
['source_month', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'payment_type', 'fare_amount', 'VendorID', 'RatecodeID', 'total_amount']
Working records: 11,198,026


## 5. Process Pickup and Drop-Off Timestamps

The timestamps are converted once and assigned to the New York timezone. The study-period restriction applies to pickup time because the target measures when demand begins.


### 5.1 Check for unusable timestamps


In [23]:
# Convert the original pickup and drop-off timestamps once.
pickup_time = pd.to_datetime(
    trips["tpep_pickup_datetime"],
    errors="coerce"
)

dropoff_time = pd.to_datetime(
    trips["tpep_dropoff_datetime"],
    errors="coerce"
)

# Assign New York local time.
# Invalid/nonexistent local clock times are marked as missing for later checking.
pickup_time = pickup_time.dt.tz_localize(
    LOCAL_TIMEZONE,
    ambiguous="NaT",
    nonexistent="NaT"
)

dropoff_time = dropoff_time.dt.tz_localize(
    LOCAL_TIMEZONE,
    ambiguous="NaT",
    nonexistent="NaT"
)

# Store the cleaned timestamps in the working dataset.
trips["pickup_time"] = pickup_time
trips["dropoff_time"] = dropoff_time

print("Pickup timestamp dtype:", trips["pickup_time"].dtype)
print("Drop-off timestamp dtype:", trips["dropoff_time"].dtype)

display(
    trips[
        [
            "tpep_pickup_datetime",
            "pickup_time",
            "tpep_dropoff_datetime",
            "dropoff_time"
        ]
    ].head()
)

Pickup timestamp dtype: datetime64[us, America/New_York]
Drop-off timestamp dtype: datetime64[us, America/New_York]


,tpep_pickup_datetime,pickup_time,tpep_dropoff_datetime,dropoff_time
0,2025-01-01 00:18:38,2025-01-01 00:18:38-05:00,2025-01-01 00:26:59,2025-01-01 00:26:59-05:00
1,2025-01-01 00:32:40,2025-01-01 00:32:40-05:00,2025-01-01 00:35:13,2025-01-01 00:35:13-05:00
2,2025-01-01 00:44:04,2025-01-01 00:44:04-05:00,2025-01-01 00:46:01,2025-01-01 00:46:01-05:00
3,2025-01-01 00:14:27,2025-01-01 00:14:27-05:00,2025-01-01 00:20:01,2025-01-01 00:20:01-05:00
4,2025-01-01 00:21:34,2025-01-01 00:21:34-05:00,2025-01-01 00:25:06,2025-01-01 00:25:06-05:00


In [24]:
invalid_timestamp = pickup_time.isna() | dropoff_time.isna()

print(f"Missing, unrecognisable or invalid local timestamps: {int(invalid_timestamp.sum()):,}")


Missing, unrecognisable or invalid local timestamps: 0


### 5.2 Check whether pickup times fall outside Q1 2025


In [ ]:
outside_study_period = (
    (trips["pickup_time"] < STUDY_START)
    | (trips["pickup_time"] >= STUDY_END)
)

print(f"Pickup timestamps outside Q1 2025: {int(outside_study_period.sum()):,}")


Pickup timestamps outside Q1 2025: 25


### 5.3 Correction: retain Q1 2025 pickups


In [ ]:
records_before = len(trips)
trips = trips.loc[~outside_study_period].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 25
Records remaining: 11,198,001


**Interpretation.** Unusable timestamps and pickups outside January-March 2025 are removed. A trip picked up on 31 March may validly finish on 1 April.


## 6. Calculate and Clean Trip Duration

Duration is a new variable, so its distribution is inspected once. Following the anchor methodology, negative, zero and positive durations shorter than 10 seconds are removed together.


### 6.1 Calculate trip duration


In [ ]:
trips["duration_seconds"] = (
    trips["dropoff_time"] - trips["pickup_time"]
).dt.total_seconds()


### 6.2 Check trip-duration values


In [ ]:
duration_summary = trips["duration_seconds"].describe(
    percentiles=[0.01, 0.50, 0.95, 0.99, 0.999]
)
display(duration_summary.to_frame("duration_seconds"))

short_duration = trips["duration_seconds"] < 10
print(f"Negative durations: {int((trips['duration_seconds'] < 0).sum()):,}")
print(f"Zero durations: {int((trips['duration_seconds'] == 0).sum()):,}")
print(
    "Positive durations below 10 seconds: "
    f"{int(((trips['duration_seconds'] > 0) & (trips['duration_seconds'] < 10)).sum()):,}"
)
print(f"Total durations below 10 seconds: {int(short_duration.sum()):,}")


,duration_seconds
count,1.119800e+07
mean,9.296689e+02
std,1.863528e+03
min,-3.088339e+06
1%,2.300000e+01
50%,7.280000e+02
95%,2.229000e+03
99%,3.678000e+03
99.9%,6.099000e+03
max,5.138800e+05


Negative durations: 298
Zero durations: 29,197
Positive durations below 10 seconds: 45,139
Total durations below 10 seconds: 74,634


### 6.3 Correction: remove durations below 10 seconds

This removes negative durations, zero durations and positive durations shorter than 10 seconds in one rule.


In [ ]:
records_before = len(trips)
trips = trips.loc[~short_duration].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 74,634
Records remaining: 11,123,367


## 7. Process Pickup and Drop-Off Locations

The already-cleaned 262-zone reference is used for both trip endpoints. No taxi-zone cleaning is repeated here.


### 7.1 Check pickup locations


In [ ]:
pickup_zone = pd.to_numeric(trips["PULocationID"], errors="coerce")
invalid_pickup_zone = ~pickup_zone.isin(valid_zone_ids)

print(f"Pickup locations outside the cleaned reference: {int(invalid_pickup_zone.sum()):,}")


Pickup locations outside the cleaned reference: 25,870


### 7.2 Correction: remove invalid pickup locations


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_pickup_zone].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 25,870
Records remaining: 11,097,497


### 7.3 Check drop-off locations


In [ ]:
dropoff_zone = pd.to_numeric(trips["DOLocationID"], errors="coerce")
invalid_dropoff_zone = ~dropoff_zone.isin(valid_zone_ids)

print(f"Drop-off locations outside the cleaned reference: {int(invalid_dropoff_zone.sum()):,}")


Drop-off locations outside the cleaned reference: 71,909


### 7.4 Correction: remove invalid drop-off locations


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_dropoff_zone].copy()
trips["LocationID"] = pd.to_numeric(
    trips["PULocationID"], errors="raise"
).astype("int16")

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 71,909
Records remaining: 11,025,588


## 8. Process Passenger Count

Explicit zero and negative values are removed. Missing passenger counts remain because missing means unknown rather than zero.


### 8.1 Check passenger count


In [ ]:
passenger_count = pd.to_numeric(trips["passenger_count"], errors="coerce")
invalid_passenger = passenger_count.notna() & passenger_count.le(0)

print(f"Missing passenger counts: {int(passenger_count.isna().sum()):,}")
print(f"Explicit zero or negative passenger counts: {int(invalid_passenger.sum()):,}")


Missing passenger counts: 2,258,618
Explicit zero or negative passenger counts: 66,603


### 8.2 Correction: remove explicit zero or negative values

Missing passenger counts are retained because they occur mainly in electronically dispatched trips and do not prove that a trip was invalid.


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_passenger].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 66,603
Records remaining: 10,958,985


## 9. Process Payment Type and Fare Amount

Explicitly voided trips and records without a positive meter fare are removed. Optional payment fields are not used as model predictors.


### 9.1 Check payment type


In [ ]:
payment_type = pd.to_numeric(trips["payment_type"], errors="coerce")
display(payment_type.value_counts(dropna=False).sort_index().rename("records").to_frame())

voided_trip = payment_type.eq(6)
print(f"Explicitly voided trips: {int(voided_trip.sum()):,}")


,records
payment_type,
0,2258618
1,7309593
2,1096626
3,64435
4,229712
5,1


Explicitly voided trips: 0


### 9.2 Check fare amount


In [ ]:
fare_amount = pd.to_numeric(trips["fare_amount"], errors="coerce")
display(fare_amount.describe(percentiles=[0.01, 0.50, 0.95, 0.99]).to_frame("fare_amount"))

invalid_fare = fare_amount.isna() | fare_amount.le(0)
print(f"Missing, zero or negative fare amounts: {int(invalid_fare.sum()):,}")


,fare_amount
count,1.095898e+07
mean,1.682571e+01
std,2.647010e+02
min,-9.770000e+02
1%,-1.000000e+01
50%,1.280000e+01
95%,4.850000e+01
99%,7.000000e+01
max,8.633721e+05


Missing, zero or negative fare amounts: 529,258


### 9.3 Correction: remove missing, zero or negative fares


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_fare].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 529,258
Records remaining: 10,429,727


## 10. Process Trip Distance

Missing, zero and negative distances are removed. Large positive distances are retained at this point because distance alone cannot show whether a trip is impossible.


### 10.1 Check trip distance


In [ ]:
trip_distance = pd.to_numeric(trips["trip_distance"], errors="coerce")
display(trip_distance.describe(percentiles=[0.01, 0.50, 0.95, 0.99, 0.999]).to_frame("trip_distance"))

invalid_distance = trip_distance.isna() | trip_distance.le(0)
print(f"Missing, zero or negative trip distances: {int(invalid_distance.sum()):,}")


,trip_distance
count,1.042973e+07
mean,5.683832e+00
std,5.321113e+02
min,0.000000e+00
1%,0.000000e+00
50%,1.710000e+00
95%,1.130000e+01
99%,1.900000e+01
99.9%,2.648000e+01
max,2.810856e+05


Missing, zero or negative trip distances: 208,929


### 10.2 Correction: remove missing, zero or negative distances


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_distance].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 208,929
Records remaining: 10,220,798


## 11. Review the Optional Diagnostic Columns

`VendorID`, `RatecodeID` and `total_amount` are examined to reveal provider patterns, special rate categories or inconsistent payment records. Vendor and rate codes do not independently remove trips. However, a missing or non-positive `total_amount` after a positive fare check represents an internally inconsistent payment record and is removed.


### 11.1 Check VendorID and RatecodeID


In [ ]:
print("VendorID distribution")
display(trips["VendorID"].value_counts(dropna=False).rename("records").to_frame())

print("RatecodeID distribution")
display(trips["RatecodeID"].value_counts(dropna=False).rename("records").to_frame())


VendorID distribution


,records
VendorID,
2,8026120
1,2193896
6,782


RatecodeID distribution


,records
RatecodeID,
1.0,8032159
NaN,1764604
2.0,266569
99.0,116760
5.0,34159
3.0,5632
4.0,913
6.0,2


**Decision.** These fields are diagnostic only. Missing `RatecodeID` values and `VendorID = 6` do not independently prove that a trip is invalid, so no records are removed here.


### 11.2 Check total amount


In [ ]:
total_amount = pd.to_numeric(trips["total_amount"], errors="coerce")
display(
    total_amount.describe(percentiles=[0.01, 0.50, 0.95, 0.99, 0.999])
    .to_frame("total_amount")
)

invalid_total_amount = total_amount.isna() | total_amount.le(0)
print(f"Missing or non-positive total amounts: {int(invalid_total_amount.sum()):,}")


,total_amount
count,1.022080e+07
mean,2.675865e+01
std,2.742238e+02
min,0.000000e+00
1%,9.340000e+00
50%,2.075000e+01
95%,7.158000e+01
99%,9.978000e+01
99.9%,1.186300e+02
max,8.633804e+05


Missing or non-positive total amounts: 69


### 11.3 Correction: remove missing or non-positive total amounts


In [ ]:
records_before = len(trips)
trips = trips.loc[~invalid_total_amount].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Records remaining: {len(trips):,}")


Records removed: 69
Records remaining: 10,220,729


**Interpretation.** The actual Q1 data contained 782 records with `VendorID = 6`, but an undocumented vendor value alone did not prove that a trip was invalid. The optional check also found 69 records with a positive fare but a non-positive total amount; these internally inconsistent payment records were removed. Large positive total amounts were retained.


## 12. Examine Distance-Duration Consistency

Large distance is assessed together with duration by calculating implied average speed. The highest-speed records are displayed with the optional diagnostic columns before a removal threshold is chosen.


### 12.1 Calculate implied average speed


In [ ]:
trips["duration_hours"] = trips["duration_seconds"] / 3600
trips["implied_speed_mph"] = trips["trip_distance"] / trips["duration_hours"]


### 12.2 Check distance-duration consistency


In [ ]:
speed_summary = trips["implied_speed_mph"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999, 0.9999]
)
display(speed_summary.to_frame("implied_speed_mph"))

speed_review_columns = [
    "source_month", "pickup_time", "dropoff_time", "duration_seconds",
    "trip_distance", "implied_speed_mph", "PULocationID", "DOLocationID",
    "fare_amount", "total_amount", "RatecodeID", "VendorID",
]
display(
    trips.nlargest(30, "implied_speed_mph")[speed_review_columns]
    .reset_index(drop=True)
)


,implied_speed_mph
count,1.022073e+07
mean,2.282274e+01
std,2.978651e+03
min,4.169949e-04
50%,9.600000e+00
90%,1.894737e+01
95%,2.404240e+01
99%,3.368421e+01
99.9%,4.434965e+01
99.99%,2.539635e+02


,source_month,pickup_time,dropoff_time,duration_seconds,trip_distance,implied_speed_mph,PULocationID,DOLocationID,fare_amount,total_amount,RatecodeID,VendorID
0,2025-01,2025-01-14 05:29:00-05:00,2025-01-14 05:31:00-05:00,120.0,124083.23,3.722497e+06,74,75,12.14,15.00,NaN,2
1,2025-01,2025-01-19 14:51:00-05:00,2025-01-19 14:58:00-05:00,420.0,276099.95,2.366571e+06,224,233,9.13,13.88,NaN,2
2,2025-01,2025-01-16 05:51:00-05:00,2025-01-16 05:54:00-05:00,180.0,106629.95,2.132599e+06,186,90,12.55,17.30,NaN,2
3,2025-02,2025-02-22 13:02:00-05:00,2025-02-22 13:04:00-05:00,120.0,61399.06,1.841972e+06,246,50,12.30,19.56,NaN,2
4,2025-01,2025-01-13 21:11:00-05:00,2025-01-13 21:17:00-05:00,360.0,181139.99,1.811400e+06,100,90,6.33,11.08,NaN,2
5,2025-02,2025-02-13 18:42:00-05:00,2025-02-13 18:46:00-05:00,240.0,117163.99,1.757460e+06,178,165,5.73,7.23,NaN,2
6,2025-03,2025-03-02 10:01:00-05:00,2025-03-02 10:10:00-05:00,540.0,253020.38,1.686803e+06,237,236,7.09,11.09,NaN,2
7,2025-03,2025-03-27 06:32:00-04:00,2025-03-27 06:42:00-04:00,600.0,281085.57,1.686513e+06,215,130,9.98,11.48,NaN,2
8,2025-01,2025-01-09 06:18:00-05:00,2025-01-09 06:24:00-05:00,360.0,143712.27,1.437123e+06,262,75,8.51,12.51,NaN,2
9,2025-03,2025-03-23 06:34:00-04:00,2025-03-23 06:37:00-04:00,180.0,66804.35,1.336087e+06,33,33,5.32,6.82,NaN,2


### 12.3 Check records above the selected speed threshold


In [ ]:
MAX_PLAUSIBLE_SPEED_MPH = 60.0
impossible_speed = trips["implied_speed_mph"] > MAX_PLAUSIBLE_SPEED_MPH

print(f"Trips above {MAX_PLAUSIBLE_SPEED_MPH:.0f} mph: {int(impossible_speed.sum()):,}")


Trips above 60 mph: 2,108


### 12.4 Correction: remove impossible distance-duration combinations


In [ ]:
records_before = len(trips)
trips = trips.loc[~impossible_speed].copy()

print(f"Records removed: {records_before - len(trips):,}")
print(f"Final valid trip records: {len(trips):,}")


Records removed: 2,108
Final valid trip records: 10,218,621


## 13. Create the Hourly Pickup Variable

After all trip-level cleaning is complete, each pickup is assigned to the beginning of its hourly interval. For example, 10:47 belongs to the interval represented by 10:00; this is not rounding to the nearest hour.


In [ ]:
trips["pickup_hour"] = trips["pickup_time"].dt.floor("h")
display(trips[["pickup_time", "pickup_hour"]].head())


,pickup_time,pickup_hour
0,2025-01-01 00:18:38-05:00,2025-01-01 00:00:00-05:00
1,2025-01-01 00:32:40-05:00,2025-01-01 00:00:00-05:00
2,2025-01-01 00:44:04-05:00,2025-01-01 00:00:00-05:00
3,2025-01-01 00:14:27-05:00,2025-01-01 00:00:00-05:00
4,2025-01-01 00:21:34-05:00,2025-01-01 00:00:00-05:00


### 13.1 Create the valid New York hourly timeline


In [ ]:
local_hour_index = pd.date_range(
    STUDY_START,
    STUDY_END,
    inclusive="left",
    freq="h",
)

print(f"Valid New York hours in Q1 2025: {len(local_hour_index):,}")
print(f"Final valid trips before aggregation: {len(trips):,}")


Valid New York hours in Q1 2025: 2,159
Final valid trips before aggregation: 10,218,621


## 14. Aggregate Valid Trips by Pickup Zone and Hour


In [ ]:
valid_trip_total = len(trips)

observed_hourly = (
    trips.groupby(["LocationID", "pickup_hour"], observed=True)
    .size()
    .rename("pickup_count")
    .reset_index()
    .sort_values(["LocationID", "pickup_hour"], ignore_index=True)
)
observed_hourly["pickup_count"] = observed_hourly["pickup_count"].astype("int32")

print(f"Valid trips represented: {valid_trip_total:,}")
print(f"Observed zone-hour combinations: {len(observed_hourly):,}")


Valid trips represented: 10,218,621
Observed zone-hour combinations: 303,285


## 15. Add Missing Zone-Hour Combinations

Every cleaned taxi zone receives one row for every valid New York hour. Unobserved combinations represent zero valid pickups.


### 15.1 Create every valid zone-hour combination


In [ ]:
backbone = pd.MultiIndex.from_product(
    [valid_zone_ids, local_hour_index],
    names=["LocationID", "pickup_hour"],
).to_frame(index=False)

print(f"Expected zone-hour rows: {len(backbone):,}")


Expected zone-hour rows: 565,658


### 15.2 Add the observed counts and fill unobserved demand with zero


In [ ]:
final_demand = backbone.merge(
    observed_hourly,
    on=["LocationID", "pickup_hour"],
    how="left",
    validate="one_to_one",
)
final_demand["was_zero_padded"] = final_demand["pickup_count"].isna()
final_demand["pickup_count"] = final_demand["pickup_count"].fillna(0).astype("int32")
final_demand["LocationID"] = final_demand["LocationID"].astype("int16")

print(f"Zero-demand rows added: {int(final_demand['was_zero_padded'].sum()):,}")
print(f"Complete zone-hour rows: {len(final_demand):,}")


Zero-demand rows added: 262,373
Complete zone-hour rows: 565,658


## 16. Create the Remaining Time Variables


In [ ]:
final_demand["date"] = final_demand["pickup_hour"].dt.strftime("%Y-%m-%d")
final_demand["hour"] = final_demand["pickup_hour"].dt.hour.astype("int8")
final_demand["day_of_week"] = final_demand["pickup_hour"].dt.dayofweek.astype("int8")
final_demand["month"] = final_demand["pickup_hour"].dt.month.astype("int8")

final_demand = final_demand[[
    "LocationID", "pickup_hour", "date", "hour", "day_of_week", "month",
    "pickup_count", "was_zero_padded",
]]

display(final_demand.head())


,LocationID,pickup_hour,date,hour,day_of_week,month,pickup_count,was_zero_padded
0,2,2025-01-01 00:00:00-05:00,2025-01-01,0,2,1,0,True
1,2,2025-01-01 01:00:00-05:00,2025-01-01,1,2,1,0,True
2,2,2025-01-01 02:00:00-05:00,2025-01-01,2,2,1,0,True
3,2,2025-01-01 03:00:00-05:00,2025-01-01,3,2,1,0,True
4,2,2025-01-01 04:00:00-05:00,2025-01-01,4,2,1,0,True


## 17. Final Dataset Summary


### 17.1 Display the final dataset summary


In [ ]:
final_summary = pd.DataFrame({
    "measure": [
        "Valid trips represented",
        "Taxi zones",
        "New York hourly timestamps",
        "Observed zone-hours",
        "Zero-demand rows added",
        "Final zone-hour rows",
        "Duplicate zone-hour records",
        "Missing pickup counts",
    ],
    "value": [
        int(final_demand["pickup_count"].sum()),
        final_demand["LocationID"].nunique(),
        final_demand["pickup_hour"].nunique(),
        int((~final_demand["was_zero_padded"]).sum()),
        int(final_demand["was_zero_padded"].sum()),
        len(final_demand),
        int(final_demand.duplicated(["LocationID", "pickup_hour"]).sum()),
        int(final_demand["pickup_count"].isna().sum()),
    ],
})
display(final_summary)


,measure,value
0,Valid trips represented,10218621
1,Taxi zones,262
2,New York hourly timestamps,2159
3,Observed zone-hours,303285
4,Zero-demand rows added,262373
5,Final zone-hour rows,565658
6,Duplicate zone-hour records,0
7,Missing pickup counts,0


## 19. Export the Integration-Ready Dataset

The final file retains the hourly demand target and the keys needed for later integration. Trip-level diagnostic fields are not exported and must not be used as prediction features.


In [ ]:
CSV_PATH = OUTPUT_DIR / "nyc_yellow_taxi_zone_hour_demand_2025_q1.csv"
final_demand.to_csv(
    CSV_PATH,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S%z",
)

print(f"Exported file: {CSV_PATH}")
print(f"Exported rows: {len(final_demand):,}")


Exported file: C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\yellow_taxi\nyc_yellow_taxi_zone_hour_demand_2025_q1.csv
Exported rows: 565,658


**Final result.** The exported dataset contains one record for every five-borough taxi-zone hour in Q1 2025. `pickup_count` is the prediction target, while `LocationID` and `pickup_hour` support later integration and zone-level evaluation.
